In [1]:
from datasets import load_dataset

dataset = load_dataset("ncbi/CellPuzzles")

/home/arism/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Found cached dataset parquet (file:///home/arism/.cache/huggingface/datasets/ncbi___parquet/ncbi--CellPuzzles-1102ae7607d753ba/0.0.0/2a3b91fbd88a2c90d1dbbb32b460cf621d31bd5b05b934492fdef7d8d6f236ec)


NotImplementedError: Loading a dataset cached in a LocalFileSystem is not supported.

In [3]:
from datasets import load_dataset
dataset = load_dataset("parquet", data_files={
    "train": "/home/arism/.cache/huggingface/datasets/ncbi___parquet/ncbi--CellPuzzles.../train/*.parquet"
})


FileNotFoundError: Unable to resolve any data file that matches '['/home/arism/.cache/huggingface/datasets/ncbi___parquet/ncbi--CellPuzzles.../train/*.parquet']' at /home/arism/cell2text/data_preprocess

In [9]:
from datasets import Dataset

# Load each split
train_dataset = Dataset.from_file(
    "/home/arism/.cache/huggingface/datasets/ncbi___parquet/ncbi--CellPuzzles-1102ae7607d753ba/0.0.0/2a3b91fbd88a2c90d1dbbb32b460cf621d31bd5b05b934492fdef7d8d6f236ec/parquet-train.arrow"
)

test_dataset = Dataset.from_file(
    "/home/arism/.cache/huggingface/datasets/ncbi___parquet/ncbi--CellPuzzles-1102ae7607d753ba/0.0.0/2a3b91fbd88a2c90d1dbbb32b460cf621d31bd5b05b934492fdef7d8d6f236ec/parquet-test.arrow"
)


print(train_dataset)
print(test_dataset)


Dataset({
    features: ['system_msg', 'user_msg', 'assistant_msg'],
    num_rows: 6912
})
Dataset({
    features: ['system_msg', 'user_msg', 'assistant_msg'],
    num_rows: 1095
})


In [6]:
print(train_dataset[42])

{'system_msg': "You are an expert assistant specialized in cell type annotation. You will be given a batch of N cells from the same donor, where each cell represents a unique cell type. For each cell, the top expressed genes are provided in descending order of expression. Using both the gene expression data and donor information, determine the correct cell type for each cell. You will also receive a list of N candidate cell types, and each candidate must be assigned to exactly one cell. Ensure that you consider all cells and candidate types together, rather than annotating each cell individually. Include your detailed reasoning within <think> and </think> tags, and provide your final answer within <answer> and </answer> tags. The final answer should be a single string listing the assigned cell types in order, separated by ' | '.", 'user_msg': 'Context: The cell is from a female at the 64-year-old stage, originating from the lung. The patient has been diagnosed with lung adenocarcinoma.

In [16]:
import re
import pickle
import pandas as pd
from pathlib import Path
from datasets import Dataset
from typing import List, Dict, Optional
import logging

logger = logging.getLogger(__name__)

class CellPuzzlesToGeneformerConverter:
    def __init__(
        self,
        token_dictionary_file: Path,
        gene_mapping_file: Optional[Path] = None,
        model_version: str = "V2",
        special_token: bool = True
    ):
        """
        Initialize converter to transform CellPuzzles dataset to Geneformer format.
        
        Parameters:
        -----------
        token_dictionary_file : Path
            Path to pickle file containing token dictionary (Ensembl IDs:token)
        gene_mapping_file : Optional[Path]
            Path to gene symbol -> Ensembl ID mapping file (CSV/TSV/pickle)
            If None, will create one from scratch (slower)
        model_version : str
            Model version ("V1" or "V2")
        special_token : bool
            Whether to add CLS and EOS tokens
        """
        self.model_version = model_version
        self.special_token = special_token
        
        # Load token dictionary (Ensembl IDs:token)
        with open(token_dictionary_file, "rb") as f:
            self.gene_token_dict = pickle.load(f)
        
        # Load or create gene mapping
        self.gene_mapping = self._load_gene_mapping(gene_mapping_file)
        
        # Check for special tokens
        if self.special_token:
            if ("<cls>" not in self.gene_token_dict.keys()) or ("<eos>" not in self.gene_token_dict.keys()):
                logger.error("<cls> and <eos> required in gene_token_dict when special_token = True.")
                raise ValueError("Special tokens missing from dictionary")
    
    def _load_gene_mapping(self, gene_mapping_file: Optional[Path]) -> Dict[str, str]:
        """Load gene symbol to Ensembl ID mapping."""
        if gene_mapping_file and gene_mapping_file.exists():
            if gene_mapping_file.suffix == '.pkl':
                with open(gene_mapping_file, 'rb') as f:
                    return pickle.load(f)
            else:
                # Assume CSV/TSV with columns: gene_symbol, ensembl_id
                df = pd.read_csv(gene_mapping_file, sep='\t' if gene_mapping_file.suffix == '.tsv' else ',')
                return dict(zip(df.iloc[:, 0], df.iloc[:, 1]))  # First col = symbol, second = ensembl
        else:
            logger.warning("No gene mapping file provided. Will create basic mapping from token dictionary.")
            # Create a basic mapping using gene symbols that might be in the token dictionary
            # This is a fallback - you should provide a proper mapping file
            return {}
    
    def parse_gene_expression(self, user_msg: str) -> List[List[str]]:
        """
        Parse gene expression data from user message.
        
        Returns:
        --------
        List of lists, where each inner list contains gene names in descending order
        """
        # Find all cell entries using regex
        cell_pattern = r'Cell \d+: (.+?)(?=Cell \d+:|Match the cells)'
        cells = re.findall(cell_pattern, user_msg, re.DOTALL)
        
        parsed_cells = []
        for cell in cells:
            # Split by comma and strip whitespace
            genes = [gene.strip() for gene in cell.split(',')]
            # Remove any trailing content after the last gene
            genes = [gene for gene in genes if gene and not gene.startswith('Match')]
            parsed_cells.append(genes)
        
        return parsed_cells
    
    def convert_genes_to_ensembl(self, gene_list: List[str]) -> List[str]:
        """
        Convert gene symbols to Ensembl IDs using local mapping.
        
        Parameters:
        -----------
        gene_list : List[str]
            List of gene symbols
            
        Returns:
        --------
        List[str]
            List of Ensembl IDs (None for genes that couldn't be converted)
        """
        if not gene_list:
            return []
        
        ensembl_ids = []
        for gene in gene_list:
            # Try exact match first
            ensembl_id = self.gene_mapping.get(gene)
            if not ensembl_id:
                # Try uppercase
                ensembl_id = self.gene_mapping.get(gene.upper())
            ensembl_ids.append(ensembl_id)
        
        return ensembl_ids
    
    def ensembl_to_tokens(self, ensembl_ids: List[str]) -> List[int]:
        """
        Convert Ensembl IDs to Geneformer tokens.
        
        Parameters:
        -----------
        ensembl_ids : List[str]
            List of Ensembl IDs
            
        Returns:
        --------
        List[int]
            List of token IDs
        """
        tokens = []
        
        # Add CLS token if specified
        if self.special_token:
            tokens.append(self.gene_token_dict.get("<cls>"))
        
        # Convert Ensembl IDs to tokens
        for ensembl_id in ensembl_ids:
            if ensembl_id and ensembl_id in self.gene_token_dict:
                tokens.append(self.gene_token_dict[ensembl_id])
            # Skip genes not in token dictionary
        
        # Add EOS token if specified
        if self.special_token:
            tokens.append(self.gene_token_dict.get("<eos>"))
        
        return tokens
    
    def process_single_example(self, example: Dict) -> Dict:
        """
        Process a single example from the CellPuzzles dataset.
        
        Parameters:
        -----------
        example : Dict
            Single example with 'user_msg', 'system_msg', 'assistant_msg'
            
        Returns:
        --------
        Dict
            Processed example with input_ids for each cell
        """
        # Parse gene expression data
        gene_lists = self.parse_gene_expression(example['user_msg'])
        
        input_ids_batch = []
        
        for genes in gene_lists:
            # Convert gene symbols to Ensembl IDs
            ensembl_ids = self.convert_genes_to_ensembl(genes)
            
            # Filter out None values
            valid_ensembl_ids = [eid for eid in ensembl_ids if eid is not None]
            
            # Convert to tokens
            tokens = self.ensembl_to_tokens(valid_ensembl_ids)
            
            input_ids_batch.append(tokens)
        
        return {
            'input_ids_batch': input_ids_batch,
            'system_msg': example['system_msg'],
            'user_msg': example['user_msg'],
            'assistant_msg': example['assistant_msg']
        }
    
    def convert_dataset(
        self, 
        dataset: Dataset, 
        batch_size: int = 100
    ) -> Dataset:
        """
        Convert entire CellPuzzles dataset to Geneformer format.
        
        Parameters:
        -----------
        dataset : Dataset
            Input CellPuzzles dataset
        batch_size : int
            Batch size for processing
            
        Returns:
        --------
        Dataset
            Converted dataset with input_ids_batch field
        """
        def process_batch(examples):
            processed = []
            for i in range(len(examples['user_msg'])):
                example = {
                    'user_msg': examples['user_msg'][i],
                    'system_msg': examples['system_msg'][i],
                    'assistant_msg': examples['assistant_msg'][i]
                }
                processed_example = self.process_single_example(example)
                processed.append(processed_example)
            
            # Reorganize for dataset format
            return {
                'input_ids_batch': [p['input_ids_batch'] for p in processed],
                'system_msg': [p['system_msg'] for p in processed],
                'user_msg': [p['user_msg'] for p in processed],
                'assistant_msg': [p['assistant_msg'] for p in processed]
            }
        
        converted_dataset = dataset.map(
            process_batch,
            batched=True,
            batch_size=batch_size,
            desc="Converting to Geneformer format"
        )
        
        return converted_dataset
    
    def save_converted_dataset(
        self, 
        converted_dataset: Dataset, 
        output_path: Path
    ):
        """
        Save converted dataset to disk.
        
        Parameters:
        -----------
        converted_dataset : Dataset
            Converted dataset
        output_path : Path
            Output path for saving
        """
        converted_dataset.save_to_disk(str(output_path))
        logger.info(f"Saved converted dataset to {output_path}")


# Usage example:
def create_gene_mapping_from_biomart():
    """
    Helper function to create gene mapping file from Ensembl BioMart.
    Run this once to create the mapping file, then reuse it.
    """
    try:
        import biomart
        
        # Connect to Ensembl BioMart
        server = biomart.BiomartServer("http://www.ensembl.org/biomart")
        hsapiens = server.datasets['hsapiens_gene_ensembl']
        
        # Query for gene symbols and Ensembl IDs
        response = hsapiens.search({
            'attributes': ['external_gene_name', 'ensembl_gene_id']
        })
        
        # Parse response and create mapping
        gene_mapping = {}
        for line in response.iter_lines():
            line = line.decode('utf-8')
            parts = line.strip().split('\t')
            if len(parts) >= 2 and parts[0] and parts[1]:
                gene_symbol = parts[0]
                ensembl_id = parts[1]
                gene_mapping[gene_symbol] = ensembl_id
        
        # Save mapping
        with open('gene_symbol_to_ensembl_mapping.pkl', 'wb') as f:
            pickle.dump(gene_mapping, f)
        
        print(f"Created gene mapping with {len(gene_mapping)} entries")
        print("Saved to: gene_symbol_to_ensembl_mapping.pkl")
        
        return gene_mapping
        
    except ImportError:
        print("biomart library not installed. Install with: pip install biomart")
        return None

def convert_cellpuzzles_dataset(
    train_dataset: Dataset,
    test_dataset: Dataset,
    token_dictionary_file: Path,
    output_dir: Path,
    gene_mapping_file: Optional[Path] = None,
    model_version: str = "V2"
):
    """
    Convert CellPuzzles train and test datasets to Geneformer format.
    
    Parameters:
    -----------
    train_dataset, test_dataset : Dataset
        Input datasets
    token_dictionary_file : Path
        Path to Geneformer token dictionary
    output_dir : Path
        Output directory
    gene_mapping_file : Optional[Path]
        Path to gene mapping file. If None, creates basic mapping.
    model_version : str
        Geneformer model version
    """
    # Initialize converter
    converter = CellPuzzlesToGeneformerConverter(
        token_dictionary_file=token_dictionary_file,
        gene_mapping_file=gene_mapping_file,
        model_version=model_version,
        special_token=True if model_version == "V2" else False
    )
    
    # Convert datasets
    print("Converting train dataset...")
    converted_train = converter.convert_dataset(train_dataset)
    
    print("Converting test dataset...")
    converted_test = converter.convert_dataset(test_dataset)
    
    # Save converted datasets
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    converter.save_converted_dataset(
        converted_train, 
        output_dir / "train_geneformer.dataset"
    )
    
    converter.save_converted_dataset(
        converted_test, 
        output_dir / "test_geneformer.dataset"
    )
    
    print(f"Conversion complete! Datasets saved to {output_dir}")
    
    # Print some statistics
    print(f"\nTrain dataset: {len(converted_train)} examples")
    print(f"Test dataset: {len(converted_test)} examples")
    
    # Show example of converted data
    if len(converted_train) > 0:
        example = converted_train[0]
        print(f"\nExample - Number of cells in first example: {len(example['input_ids_batch'])}")
        if len(example['input_ids_batch']) > 0:
            print(f"First cell tokens length: {len(example['input_ids_batch'][0])}")
            print(f"First few tokens: {example['input_ids_batch'][0][:10]}")

# Example usage:
# 1. First, create gene mapping (run once):
# create_gene_mapping_from_biomart()

# 2. Then convert datasets:
# convert_cellpuzzles_dataset(
#     train_dataset=train_dataset,
#     test_dataset=test_dataset, 
#     token_dictionary_file=Path("/path/to/geneformer_token_dict.pkl"),
#     gene_mapping_file=Path("gene_symbol_to_ensembl_mapping.pkl"),
#     output_dir=Path("./converted_datasets"),
#     model_version="V2"
# )

In [18]:
create_gene_mapping_from_biomart()

Created gene mapping with 41164 entries
Saved to: gene_symbol_to_ensembl_mapping.pkl


{'MT-TF': 'ENSG00000210049',
 'MT-RNR1': 'ENSG00000211459',
 'MT-TV': 'ENSG00000210077',
 'MT-RNR2': 'ENSG00000210082',
 'MT-TL1': 'ENSG00000209082',
 'MT-ND1': 'ENSG00000198888',
 'MT-TI': 'ENSG00000210100',
 'MT-TQ': 'ENSG00000210107',
 'MT-TM': 'ENSG00000210112',
 'MT-ND2': 'ENSG00000198763',
 'MT-TW': 'ENSG00000210117',
 'MT-TA': 'ENSG00000210127',
 'MT-TN': 'ENSG00000210135',
 'MT-TC': 'ENSG00000210140',
 'MT-TY': 'ENSG00000210144',
 'MT-CO1': 'ENSG00000198804',
 'MT-TS1': 'ENSG00000210151',
 'MT-TD': 'ENSG00000210154',
 'MT-CO2': 'ENSG00000198712',
 'MT-TK': 'ENSG00000210156',
 'MT-ATP8': 'ENSG00000228253',
 'MT-ATP6': 'ENSG00000198899',
 'MT-CO3': 'ENSG00000198938',
 'MT-TG': 'ENSG00000210164',
 'MT-ND3': 'ENSG00000198840',
 'MT-TR': 'ENSG00000210174',
 'MT-ND4L': 'ENSG00000212907',
 'MT-ND4': 'ENSG00000198886',
 'MT-TH': 'ENSG00000210176',
 'MT-TS2': 'ENSG00000210184',
 'MT-TL2': 'ENSG00000210191',
 'MT-ND5': 'ENSG00000198786',
 'MT-ND6': 'ENSG00000198695',
 'MT-TE': 'ENSG00000

In [20]:
convert_cellpuzzles_dataset(
    train_dataset=train_dataset,
    test_dataset=test_dataset,
    token_dictionary_file=Path("/home/arism/cell2text/data_preprocess/Geneformer/geneformer/token_dictionary_gc104M.pkl"),
    gene_mapping_file=Path("gene_symbol_to_ensembl_mapping.pkl"),  # Pre-created file
    output_dir=Path("./converted_datasets")
)

Converting train dataset...


Converting test dataset...


Conversion complete! Datasets saved to converted_datasets

Train dataset: 6912 examples
Test dataset: 1095 examples

Example - Number of cells in first example: 10
First cell tokens length: 42
First few tokens: [2, 11717, 13413, 13467, 12493, 5525, 3187, 1239, 16581, 6676]


In [21]:
print(test_dataset[0]['user_msg'])

Context: The cell is from a female at the 73-year-old stage, originating from the lung. The patient has been diagnosed with chronic obstructive pulmonary disease. The patient is a smoker. There is no cancer present. 

Cell 1: MT2A, ACTB, MT1X, MTATP6P29, MYL9, MTND4LP30, CRIP1, DSTN, MTND2P13, MTCO2P22, S100A6, MTCYBP19, MALAT1, VIM, RPLP1, RGS5, TPT1, LGALS1, TPM2, MTND3P6, MTND1P22, PTMA, TMSB4X, STEAP1B, MT1M, LPP, RPL21, RPL32, HSPB1, RPL18A, RPL10, EEF1A1, RBFOX2, RPL41, IFITM3, CTNNB1, MYH11, RPL34, RPS3A, RPS27, CALD1, IL6ST, RPS18, RPS27L, IGFBP7, B2M, TPM1, DTWD2, MYL6, RPL39
Cell 2: MALAT1, FTL, MTCO2P22, TMSB4X, B2M, MTND4LP30, IL6ST, RPS19, RBFOX2, CCSER1, RPL41, RPS27, RPL10, ACTB, MTATP6P29, MTND2P13, RPS12, STEAP1B, RPL13A, S100A4, RPL34, TMSB10, RPL28, RPL32, RPL39, RPL13, RPL21, RPS15A, RPL36A, DGKI, SAT1, RPS2, RPS29, PLEKHA7, DUSP1, EEF1A1, RPS17, RPS4X, RPLP1, RPS6, RPS9, TYROBP, NAMPT, MTCYBP19, RPS3A, S100A6, RPL27A, MTND3P6, RPS16, RPLP2
Cell 3: SCGB3A1, SCGB1A1,

In [23]:
geneformer_train_dataset = Dataset.load_from_disk(
    "/home/arism/cell2text/data_preprocess/converted_datasets/train_geneformer.dataset"
)
geneformer_test_dataset = Dataset.load_from_disk(
    "/home/arism/cell2text/data_preprocess/converted_datasets/test_geneformer.dataset"
)

In [24]:
print(geneformer_train_dataset)

Dataset({
    features: ['system_msg', 'user_msg', 'assistant_msg', 'input_ids_batch'],
    num_rows: 6912
})


In [38]:
print(len(geneformer_train_dataset[42]["input_ids_batch"][0]))

41


In [39]:
print(geneformer_train_dataset[0]["user_msg"])

Context: The cell is from a female at the 66-year-old stage, originating from the lung. The patient is healthy with no diagnosed disease. The patient is a non-smoker. There is no cancer present. 

Cell 1: MALAT1, B2M, MTND4LP30, NEAT1, ANKRD36C, MTND3P6, DNAH12, ALCAM, MTND2P13, ATXN1, DNAH11, MAP3K13, CFAP299, LRRIQ1, MTCO2P22, ANXA2, MTATP6P29, CLDN4, HSP90AA1, MTND1P22, CFAP43, MTCYBP19, HLA-B, TMC5, EFNA5, IL6ST, RAB11FIP1, MDM2, BTBD9, AGBL4, MAPK10, PDE4D, LRBA, IGFBP7, FGF13, FOXP1, RABGAP1L, CAPS, MAGI1, CFAP54, MACF1, VMP1, EVA1C, EIF4G3, GSTP1, PTPRT, RPL34, CAMK1D, NFIA, MECOM
Cell 2: MALAT1, B2M, LINC-PINT, RPS27, RPL13A, RPL10, RPL39, RPL28, RPS29, RPL32, RPL34, RPS2, PTMA, RPL3, RPS19, RPS18, RPS15A, RPL27A, RPL13, TMSB10, RPLP2, RPL15, AKAP13, IL6ST, STEAP1B, RPS4X, RPS27A, RPS12, RPL18A, TPT1, RPS15, RPL14, CHST11, RPS14, RPL19, RPL21, CNOT6L, TMSB4X, RPS6, RPS16, RPS17, RPLP1, RPL36, RPL31, CD44, RPL12, RPL11, UBA52, MBNL1, HLA-A
Cell 3: MALAT1, TMSB4X, B2M, FTL, SAT1,